In [ ]:
import os
import re
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from datetime import datetime
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from scipy.stats import linregress
from pathlib import Path
from abc import ABCMeta, abstractmethod
from time import time
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression

In [ ]:
sys.path.append(os.path.abspath('..'))
from configs.config import *
from src.util import Logger, Util

In [ ]:
# import importlib
# import configs.config
# importlib.reload(configs.config)
# from configs.config import *

In [ ]:
pd.set_option("display.max_columns",500)
pd.set_option("display.max_rows", 500)

# 基底クラス

In [ ]:
def decorate(s: str, decoration=None):
    if decoration is None:
        decoration = '★' * 20

    return ' '.join([decoration, str(s), decoration])

class Timer:
    def __init__(self, logger=None, format_str='{:.3f}[s]', prefix=None, suffix=None, sep=' ', verbose=0):

        if prefix: format_str = str(prefix) + sep + format_str
        if suffix: format_str = format_str + sep + str(suffix)
        self.format_str = format_str
        self.logger = logger
        self.start = None
        self.end = None
        self.verbose = verbose

    @property
    def duration(self):
        if self.end is None:
            return 0
        return self.end - self.start

    def __enter__(self):
        self.start = time()

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time()
        if self.verbose is None:
            return
        out_str = self.format_str.format(self.duration)
        if self.logger:
            self.logger.info(out_str)
        else:
            print(out_str)

In [ ]:
class FeatureBase(metaclass=ABCMeta):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        self.use_cache = use_cache
        self.name = self.__class__.__name__
        self.cache_dir = Path(DIR_FEATURE)
        self.logger = logger
        self.seve_cache = save_cache
        self.use_cols = None
        self.key_column = None
    
    # 共通のキー整形 & 重複チェック
    def enforce_key_integrity(self, df: pd.DataFrame) -> pd.DataFrame:
        for key in self.key_column:
            if key not in df.columns:
                raise KeyError(f"{self.name}: キーカラム '{key}' が存在しません")
        assert ~df[self.key_column].duplicated().any(), f"{self.name}: 主キー {self.key_column} に重複があります"
    
    @abstractmethod
    def _create_feature(self) -> pd.DataFrame:
        """
        特徴量生成の実装をサブクラスで定義する必要があります。
        :return: pd.DataFrame 生成された特徴量
        """
        raise NotImplementedError()

    # 特徴量生成処理
    def create_feature(self) -> pd.DataFrame:

        # クラス名.pkl
        file_name = os.path.join(self.cache_dir, f"{self.name}.pkl")

        # キャッシュを使う & ファイルがあるなら読み出し
        if os.path.isfile(str(file_name)) and self.use_cache:
            feature = pd.read_pickle(file_name)

        # 変換処理を実行
        else:
            # train/testの区別なく変換処理を実行
            feature = self._create_feature()

            # 主キーチェック
            if self.key_column is not None:
                self.enforce_key_integrity(feature)

            # 保存する場合
            if self.seve_cache:
                feature.to_pickle(file_name)

        return feature

In [ ]:
def one_hot_encode(df, col, drop_col=True):
    """
    特定の列に対してOne-Hotエンコーディングを適用します。
    
    :param df: pd.DataFrame 対象のDataFrame
    :param col: str エンコードする列名
    :return: pd.DataFrame エンコードされたDataFrame
    """
    # Initialize OneHotEncoder
    encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

    # Fit and transform the specified column
    encoded = encoder.fit_transform(df[[col]])

    # Convert the encoded array to a DataFrame
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out([col]))

    # concat
    df_concat = pd.concat([df, encoded_df], axis=1)

    # drop 
    if drop_col:
        df_concat.drop(columns=col, inplace=True)

    return df_concat

In [ ]:
def clean_feature_names(data):
    # 特徴量名を修正
    data.columns = data.columns.str.replace(r'[^\w]', '_', regex=True)
    return data

# 特殊文字をアンダースコアに置換
def replace_special_characters(text):
    """
    特徴量名から特殊文字を削除し、LightGBMがサポートする形式に変換する。
    例："列名@!#" → "列名___"
    """
    return re.sub(r'[^\w]', '_', text)

# 継承クラス

In [ ]:
class Key(FeatureBase):
    """
    TrainFeatureクラスは、train.csvデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号', 'category']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        train.csvデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # train.csvデータを読み込む
        df_train = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_train.pkl'))
        df_test = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_test.pkl'))
        df_Key = pd.concat([df_train, df_test], ignore_index=True)[self.key_column]

        return df_Key

In [ ]:
class Target(FeatureBase):
    """
    Targetクラスは、ターゲットデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号', 'category']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        ターゲットデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成されたターゲットデータを含むDataFrame。
        """
        # ターゲットデータを読み込む
        df_train = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_train.pkl'))

        # 必要なカラムを選択
        df_target = df_train[['社員番号', 'category', 'target']]

        # 主キーとターゲット列を含むDataFrameを返す
        return df_target

In [ ]:
class CategoryFeature(FeatureBase):
    """
    TrainFeatureクラスは、train.csvデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['category']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        train.csvデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # train.csvデータを読み込む
        df_train = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_train.pkl'))
        df_test = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_test.pkl'))
        df_all = pd.concat([df_train, df_test], ignore_index=True)

        df_category_feature = df_all.copy().drop_duplicates('category')[self.key_column]

        # One-hotエンコーディング
        # df_category_feature = one_hot_encode(df_category_feature, 'category', False)

        # labaelエンコーディング
        le = LabelEncoder()
        df_category_feature['le_category'] = le.fit_transform(df_category_feature['category'])

        # 主キーとターゲット列を含むDataFrameを返す
        return df_category_feature

In [ ]:
class CareerFeature(FeatureBase):
    """
    CareerBlockクラスは、キャリアデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        キャリアデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 前処理済みのキャリアデータを読み込む
        df_career = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_career.pkl"))

        df_career_feature = df_career.copy()

        # ポジティブな回答数
        df_career_feature['positive_responses'] = df_career_feature.iloc[:, 1:].apply(lambda row: (row>=4).sum(), axis=1)

        # ネガティブな回答数
        df_career_feature['negative_responses'] = df_career_feature.iloc[:, 1:].apply(lambda row: (row<=2).sum(), axis=1)

        # ポジティブな回答の割合
        df_career_feature['positive_ratio'] = df_career_feature['positive_responses'] / (df_career_feature.shape[1] - 1)

        return df_career_feature

In [ ]:
   
class UdemyActivityFeature(FeatureBase):
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:

        # 前処理済みのUdemy活動データを読み込む
        df_udemy = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_udemy_activity.pkl"))

        df_udemy_feature = df_udemy.copy()[self.key_column].drop_duplicates()

        # クイズ判定
        df_udemy["is_quiz"] = df_udemy["レクチャーもしくはクイズ"]=="Quiz"

        # 基本統計量の集計
        df_udemy_activity_numerical = df_udemy.groupby(self.key_column).agg(
            # 推定完了率%
            mean_推定完了率=('推定完了率_', 'mean'),
            min_推定完了率=('推定完了率_', 'min'),
            max_推定完了率=('推定完了率_', 'max'),
            std_推定完了率=('推定完了率_', 'std'),
            count_推定完了率=('推定完了率_', 'count'),
            # 最終結果（クイズの場合）
            mean_最終結果=('最終結果_クイズの場合_', 'mean'),
            min_最終結果=('最終結果_クイズの場合_', 'min'),
            max_最終結果=('最終結果_クイズの場合_', 'max'),
            std_最終結果=('最終結果_クイズの場合_', 'std'),
            count_最終結果=('最終結果_クイズの場合_', 'count'),
            # マーク済み修了
            mean_マーク済み修了=('マーク済み修了', 'mean'),
            min_マーク済み修了=('マーク済み修了', 'min'),
            max_マーク済み修了=('マーク済み修了', 'max'),
            std_マーク済み修了=('マーク済み修了', 'std'),
            count_マーク済み修了=('マーク済み修了', 'count'),
        )

        # 正規化の結果同じ値になったものを分別
        map_val = {val: f'{i}_{val}' for i, val in enumerate(df_udemy['コースカテゴリー'].unique())}
        df_udemy['コースカテゴリー'] = df_udemy['コースカテゴリー'].map(lambda x: map_val.get(x, np.nan))
        # コースカテゴリごとの回数を集計
        # 正規化の結果同じ値になったものを分別
        map_val = {val: f'{i}_{val}' for i, val in enumerate(df_udemy['コースカテゴリー'].unique())}
        df_udemy['コースカテゴリー'] = df_udemy['コースカテゴリー'].map(lambda x: map_val.get(x, np.nan))
        # 集計
        df_udemy_activity_course_category = df_udemy.pivot_table(
            index='社員番号',
            columns='コースカテゴリー',
            values='コースID',
            aggfunc='count',
            fill_value=None,
        ).reset_index()
        # カラム名を変更
        prefix = "ua_カテゴリ_"
        df_udemy_activity_course_category.columns = [col if col=='社員番号' else prefix + col for col in df_udemy_activity_course_category.columns]

        # レクチャーもしくはクイズごとの回数を集計
        df_udemy_activity_type = df_udemy.pivot_table(
            index='社員番号',
            columns='レクチャーもしくはクイズ',
            values='コースID',
            aggfunc='count',
            fill_value=None,
        ).reset_index()
        # カラム名を変更
        prefix = "ua_レクチャーorクイズ_"
        df_udemy_activity_type.columns = [col if col=='社員番号' else prefix + col for col in df_udemy_activity_type.columns]

        # # コースIDごとの回数を集計
        # df_udemy_activity_course_id = df_udemy.pivot_table(
        #     index='社員番号',
        #     columns='コースID',
        #     values='コースID',
        #     aggfunc='count',
        #     fill_value=0
        # ).reset_index()     
        # # カラム名を変更
        # prefix = "ua_コースID_"
        # df_udemy_activity_course_id.columns = [str(col[0]) if col[0]=='社員番号' else prefix + str(col[1]) for col in df_udemy_activity_course_id.columns]

        # df_udemy_feature = df_udemy.groupby(self.key_column).agg(
        #     count_コースID=('コースID', 'count'),
        #     nunique_コースID=('コースID', 'nunique'),
        #     nunique_コースタイトル=('コースタイトル', 'nunique'),
        #     nunique_コースカテゴリ=('コースカテゴリー', 'nunique'),
        #     nunique_学習日数=('開始日', pd.Series.nunique),
        #     sum_マーク済み修了=('マーク済み修了', 'sum'),
        #     mean_推定完了率=('推定完了率%', 'mean'),
        #     min_開始日=('開始日', 'min'),
        #     max_開始日=('開始日', 'max'),
        #     rate_Quiz=('is_quiz', 'mean'),
        # ).reset_index()

        # # 学習スパン（日数）
        # df_udemy_feature["learning_span"] = (df_udemy_feature["max_開始日"] - df_udemy_feature["min_開始日"]).dt.days
        # # 日付型を数値型に変換
        # df_udemy_feature["min_開始日"] = df_udemy_feature["min_開始日"].apply(lambda x: float(datetime.strftime(x, format='%Y%m%d')))
        # df_udemy_feature["max_開始日"] = df_udemy_feature["max_開始日"].apply(lambda x: float(datetime.strftime(x, format='%Y%m%d')))

        # # クイズスコアの集計
        # df_quiz = df_udemy[df_udemy["is_quiz"]].copy()
        # df_quiz_stats = df_quiz.groupby(self.key_column).agg(
        #     count_クイズ=('最終結果（クイズの場合）', 'count'),
        #     mean_クイズスコア=('最終結果（クイズの場合）', 'mean'),
        # ).reset_index()

        # カラム名を変更
        # df_udemy_activity_course_category.columns = [replace_special_characters(col) for col in df_udemy_activity_course_category.columns]
        # df_udemy_activity_type.columns = [replace_special_characters(col) for col in df_udemy_activity_type.columns]


        # マージ
        df_udemy_feature = df_udemy_feature.merge(df_udemy_activity_numerical, on=self.key_column, how='left')
        df_udemy_feature = df_udemy_feature.merge(df_udemy_activity_course_category, on=self.key_column, how='left')
        df_udemy_feature = df_udemy_feature.merge(df_udemy_activity_type, on=self.key_column, how='left')

        # カラム名の修正
        df_udemy_feature = clean_feature_names(df_udemy_feature)

        return df_udemy_feature

class UdemyEmbedding(FeatureBase):
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        Udemyのコースタイトルの埋め込み特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 前処理済みのUdemy活動データを読み込む
        df_udemy = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_udemy_activity.pkl"))

        df_udemy_embeddings_feature = df_udemy.copy()[self.key_column].drop_duplicates()

        def create_sparse_matrix(df: pd.DataFrame, user_col: str, action_col: str, value_col=None,) -> tuple[sp.csr_matrix, LabelEncoder, LabelEncoder]:
            # user_col と action_col を数値に変更する
            user_encoder = LabelEncoder()
            action_encoder = LabelEncoder()
            user_array = user_encoder.fit_transform(df[user_col].to_numpy().ravel())
            action_array = action_encoder.fit_transform(df[action_col].to_numpy().ravel())

            # 重みを指定する (value_colがNoneの場合は1を指定)
            data_array = df[value_col].to_numpy().ravel() if value_col is not None else np.ones(len(df))

            # スパース行列を作成する
            sparse_matrix = sp.csr_matrix(
                (data_array, (user_array, action_array)),
                shape=(len(user_encoder.classes_), len(action_encoder.classes_)),
            )
            return sparse_matrix, user_encoder, action_encoder

        # スパース行列を作成
        sparse_matrix, user_encoder, action_encoder = create_sparse_matrix(df_udemy, "社員番号", "コースタイトル")
        # SVDで次元削減
        n_components = 8
        svd = TruncatedSVD(n_components=n_components, random_state=42)

        # 社員番号の埋め込み
        user_embeddings = svd.fit_transform(sparse_matrix)
        # DFとして整形
        df_udemy_user_embeddings = pd.concat([
            pd.DataFrame({"社員番号": user_encoder.classes_}),
            pd.DataFrame(user_embeddings, columns=[f'svd_コースタイトル_{i}' for i in range(user_embeddings.shape[1])])
        ], axis=1)

        # コースタイトルの埋め込み
        action_embeddings = svd.components_.T
        # コースタイトル → 埋め込みマップを構築
        course_title_to_vec = {
            course: action_embeddings[idx]
            for course, idx in zip(action_encoder.classes_, range(len(action_encoder.classes_)))
        }
        # 各社員ごとに受講コースのベクトル平均を計算
        def compute_mean_embedding(group):
            embeddings = [course_title_to_vec[title] for title in group['コースタイトル'] if title in course_title_to_vec]
            if embeddings:
                return pd.Series(np.mean(embeddings, axis=0))
            else:
                return pd.Series([np.nan] * n_components)
        df_mean_embeddings = df_udemy.groupby("社員番号").apply(compute_mean_embedding).reset_index()
        df_mean_embeddings.columns = ["社員番号"] + [f"mean_svd_コースタイトル_{i}" for i in range(n_components)]

        # マージ
        df_udemy_embeddings_feature = df_udemy_embeddings_feature.merge(df_udemy_user_embeddings, on=self.key_column, how='left')
        df_udemy_embeddings_feature = df_udemy_embeddings_feature.merge(df_mean_embeddings, on=self.key_column, how='left')

        return df_udemy_embeddings_feature

In [ ]:
class DxFeature(FeatureBase):
    """
    DxFeatureクラスは、DX関連のデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        DX関連データを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        df_dx = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_dx.pkl"))

        df_dx_feature = df_dx.copy()[self.key_column].drop_duplicates()

        # 基本集計値
        df_dx_numerical = df_dx.groupby(self.key_column).agg(
            dx_count=('研修名', 'count'),
            dx_unique_研修名=('研修名', 'nunique'),
            dx_unique_研修カテゴリ=('研修カテゴリ', 'nunique'),
        ).reset_index()

        # 研修カテゴリごとの参加回数を集計
        df_category_count = df_dx.pivot_table(
            index='社員番号',
            columns="研修カテゴリ",
            values="研修実施日",
            aggfunc="count",
            fill_value=None,
        ).reset_index()
        # カラム名の変更
        prefix = "dx_研修カテゴリ_"
        df_category_count.columns = [col if col=='社員番号' else prefix + col for col in df_category_count.columns]

        # 研修名ごとの参加回数を集計
        df_name_count = df_dx.pivot_table(
            index='社員番号',
            columns="研修名",
            values="研修実施日",
            aggfunc="count",
            fill_value=None,
        ).reset_index()
        # カラム名の変更
        prefix = "dx_研修名_"
        df_name_count.columns = [col if col=='社員番号' else prefix + col for col in df_name_count.columns]

        # 各社員の研修参加回数
        # df_dx_feature = df_dx.groupby(self.key_column).agg(
        #     count=('研修名', 'count'),
        #     unique_training_count=('研修名', 'nunique'),
        # ).reset_index()

        # 各社員のユニークな研修カテゴリ数
        # df_dx_feature['unique_training_categories'] = dx_data.groupby(self.key_column)['研修カテゴリ'].transform('nunique')

        # マージ
        df_dx_feature = df_dx_feature.merge(df_dx_numerical, on=self.key_column, how='left')
        df_dx_feature = df_dx_feature.merge(df_category_count, on=self.key_column, how='left')
        df_dx_feature = df_dx_feature.merge(df_name_count, on=self.key_column, how='left')

        # カラム名の修正
        df_dx_feature = clean_feature_names(df_dx_feature)

        return df_dx_feature

In [ ]:
class HrFeature(FeatureBase):
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger=None)
        self.key_column = ['社員番号']

    def _create_feature(self) -> pd.DataFrame:

        df_hr = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_hr.pkl'))

        df_hr_feature = df_hr.copy()[self.key_column].drop_duplicates()

        # # 実施期間を算出（日数）
        # df_hr['研修日数'] = (df_hr['実施終了日'] - df_hr['実施開始日']).dt.days + 1
        # df_hr['研修日数'] = df_hr['研修日数'].fillna(1).clip(lower=1)

        # # 基礎集計値
        # df_hr_numetric = df_hr.groupby(self.key_column).agg(
        #     hr_count=('研修名', 'count'),
        #     hr_unique_研修名=('研修名', 'nunique'),
        #     hr_unique_カテゴリ=('カテゴリ', 'nunique'),
        #     min_実施開始日=("実施開始日", "min"),
        #     max_実施終了日=("実施終了日", "max"),
        #     sum_研修日数=("研修日数", "sum"),
        # ).reset_index()

        # # 活動期間（最終日 - 初日）
        # df_hr_numetric["hr_active_days"] = (df_hr_numetric["max_実施終了日"] - df_hr_numetric["min_実施開始日"]).dt.days
        
        # # 日付型を数値型に変換
        # df_hr_numetric["min_実施開始日"] = df_hr_numetric["min_実施開始日"].apply(lambda x: float(datetime.strftime(x, format='%Y%m%d')))
        # df_hr_numetric["max_実施終了日"] = df_hr_numetric["max_実施終了日"].apply(lambda x: float(datetime.strftime(x, format='%Y%m%d')))

        # 研修カテゴリごとの参加回数を集計
        df_category_count = df_hr.pivot_table(
            index='社員番号',
            columns="カテゴリ",
            values="実施開始日",
            aggfunc="count",
            fill_value=None,
        ).reset_index() 
        # カラム名の変更
        prefix = "hr_研修カテゴリ_"
        df_category_count.columns = [col if col=='社員番号' else prefix + col for col in df_category_count.columns]

        # 研修名ごとの参加回数を集計
        df_name_count = df_hr.pivot_table(
            index='社員番号',
            columns="研修名",
            values="実施開始日",
            aggfunc="count",
            fill_value=None,
        ).reset_index()
        # カラム名の変更
        prefix = "hr_研修名_"
        df_name_count.columns = [col if col=='社員番号' else prefix + col for col in df_name_count.columns]

        # # 実施期間を算出（日数）
        # df_hr['研修日数'] = (df_hr['実施終了日'] - df_hr['実施開始日']).dt.days + 1
        # df_hr['研修日数'] = df_hr['研修日数'].fillna(1).clip(lower=1)

        # # 特徴量作成
        # df_hr_feature = df_hr.groupby("社員番号").agg(
        #     n_hr_total=("研修名", "count"),
        #     n_hr_unique_program=("研修名", "nunique"),
        #     n_hr_unique_category=("カテゴリ", "nunique"),
        #     first_hr_date=("実施開始日", "min"),
        #     last_hr_date=("実施終了日", "max"),
        #     n_hr_days=("研修日数", "sum"),
        # ).reset_index()

        # # 活動期間（最終日 - 初日）
        # df_hr_feature["hr_active_days"] = (df_hr_feature["last_hr_date"] - df_hr_feature["first_hr_date"]).dt.days
        # df_hr_feature.drop(["first_hr_date", "last_hr_date"], axis=1, inplace=True)

        # マージ
        # df_hr_feature = df_hr_feature.merge(df_hr_numetric, on=self.key_column, how='left')
        df_hr_feature = df_hr_feature.merge(df_category_count, on=self.key_column, how='left')
        df_hr_feature = df_hr_feature.merge(df_name_count, on=self.key_column, how='left')

        return df_hr_feature

In [ ]:
class OvertimeWorkByMonthFeature(FeatureBase):
    """
    OvertimeWorkFeatureクラスは、月ごとの残業データを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        残業データを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 残業データを読み込む
        df_overtime = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_overtime_work_by_month.pkl"))

        # 基礎特徴量
        df_overtime_base_feature = df_overtime.groupby(self.key_column).agg(
            avg_overtime=('hours', 'mean'),
            median_overtime=('hours', 'median'),
            max_overtime=('hours', 'max'),
            min_overtime=('hours', 'min'),
            # total_overtime_hours=('hours', 'sum'),
            std_overtime=('hours', 'std'),
            count_overtime_months=('hours', 'count'),
        ).reset_index()


        return df_overtime_base_feature
    
class OvertimeWorkByMonthTimeseriesFeature(FeatureBase):
    """
    OvertimeWorkByMonthTimeseriesFeatureクラスは、月ごとの残業データの時系列特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        残業データの時系列特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 残業データを読み込む
        df_overtime = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_overtime_work_by_month.pkl"))

        # dateの欠損行はNullで埋める
        unique_employees = df_overtime['社員番号'].unique()
        unique_dates = df_overtime['date'].unique()
        all_combinations = pd.MultiIndex.from_product([unique_employees, unique_dates], names=['社員番号', 'date']).to_frame(index=False)
        df_all_date_overtime = pd.merge(all_combinations, df_overtime, on=['社員番号', 'date'], how='left')

        # lag特徴量
        def make_worker_hours_lag_features(df_overtime, lag=35):
            """
            社員別の過去労働時間（lag特徴量）を作成し、最新月の1行にまとめる。

            Parameters:
                df_overtime: DataFrame
                    '社員番号', 'date', 'hours' を含むDataFrame
                lag: int
                    生成する最大lag数（例：35であれば hours_1_age ～ hours_35_age）
            Returns:
                df_worker_lag: DataFrame
                    社員番号ごとの最新行 + lag特徴量（hours_0_age ～ hours_{lag}_age）
            """
            df = df_overtime.copy()
            df = df.sort_values(['社員番号', 'date']).reset_index(drop=True)

            # lag特徴量を生成
            for i in range(1, lag + 1):
                df[f'hours_{i}_age'] = df.groupby('社員番号')['hours'].shift(i)

            # 最新行を抽出
            df_worker_lag = df.groupby('社員番号').tail(1).reset_index(drop=True)

            # カラム整形
            lag_cols = [f'hours_{i}_age' for i in range(1, lag + 1)]
            df_worker_lag = df_worker_lag[['社員番号', 'date', 'hours'] + lag_cols]
            df_worker_lag = df_worker_lag.rename(columns={'hours': 'hours_0_age'})

            return df_worker_lag
        df_worker_lag = make_worker_hours_lag_features(df_all_date_overtime, lag=35)
        df_worker_lag.drop('date', axis=1, inplace=True)
        df_worker_lag.head()


        # 移動特徴量を生成
        # 各種統計特徴量を生成するウィンドウサイズのリスト
        windows = [2, 3, 4, 5, 6, 9, 12, 15, 18, 24, 36]
        current_hours = df_worker_lag['hours_0_age']

        for w in windows:
            # 直近 w ヶ月分の hours列（hours_0_age ～ hours_{w-1}_age）
            cols = [f'hours_{i}_age' for i in range(0, w)]

            # 移動統計量を計算
            df_worker_lag[f'hours_ma_{w}'] = df_worker_lag[cols].mean(axis=1)              # 平均
            df_worker_lag[f'hours_std_{w}'] = df_worker_lag[cols].std(axis=1)              # 標準偏差
            df_worker_lag[f'hours_max_{w}'] = df_worker_lag[cols].max(axis=1)              # 最大値
            df_worker_lag[f'hours_min_{w}'] = df_worker_lag[cols].min(axis=1)              # 最小値
            df_worker_lag[f'hours_diff_ma_{w}'] = current_hours - df_worker_lag[f'hours_ma_{w}']  # 今月と平均の差
            df_worker_lag[f'hours_range_{w}'] = df_worker_lag[f'hours_max_{w}'] - df_worker_lag[f'hours_min_{w}']  # 振れ幅
            df_worker_lag[f'hours_missing_count_{w}'] = df_worker_lag[cols].isna().sum(axis=1)  # 欠損数
            df_worker_lag[f'hours_zscore_{w}'] = (current_hours - df_worker_lag[f'hours_ma_{w}']) / (df_worker_lag[f'hours_std_{w}'] + 1e-6)  # z-score

            # 今月と wヶ月前との比較（差分）
            df_worker_lag[f'hours_diff_prev_{w}'] = current_hours - df_worker_lag[f'hours_{w-1}_age']

            # 線形トレンド（回帰直線の傾き）を算出
            trends = []
            for _, row in df_worker_lag[cols].iterrows():
                y = row.values
                x = np.arange(1, w + 1).reshape(-1, 1)
                if np.isnan(y).all():
                    trends.append(np.nan)
                else:
                    mask = ~np.isnan(y)
                    reg = LinearRegression().fit(x[mask], y[mask])
                    trends.append(reg.coef_[0])
            df_worker_lag[f'hours_trend_{w}'] = trends

            # 今月が過去平均より ±30% を超えているか
            ratio = current_hours / (df_worker_lag[f'hours_ma_{w}'] + 1e-6)
            df_worker_lag[f'hours_over_{w}_flag'] = (ratio > 1.3).astype(int)   # 今月が30%以上多い
            df_worker_lag[f'hours_under_{w}_flag'] = (ratio < 0.7).astype(int)  # 今月が30%以上少ない

            # 指数移動平均（最近の値をより重視した平均）
            df_worker_lag[f'hours_ewm_{w}'] = df_worker_lag[cols].T.ewm(span=3, axis=0).mean().iloc[-1]
        
        
        # 前月との労働時間差が閾値を超えた回数
        spike_threshold = 20
        spike_count = []

        for _, row in df_worker_lag.iterrows():
            diffs = []
            for i in range(1, 35):
                col_now = f'hours_{i}_age'
                col_next = f'hours_{i+1}_age'
                if pd.notna(row[col_now]) and pd.notna(row[col_next]):
                    if abs(row[col_now] - row[col_next]) > spike_threshold:
                        diffs.append(1)
            spike_count.append(sum(diffs))

        df_worker_lag['hours_spike_count'] = spike_count

        def direction_mode(row, window=6):
            """
            過去 window ヶ月分の労働時間（hours_{i}_age）を比較し、
            増減方向の多数決に基づいてトレンド方向を判定する。

            Parameters:
                row: pd.Series
                window: int (比較対象の月数)
            Returns:
                int: -1 / 0 / 1（下降 / 中立 / 上昇）
            """
            directions = []
            for i in range(0, window - 1):
                a, b = row.get(f'hours_{i}_age'), row.get(f'hours_{i+1}_age')
                if pd.notna(a) and pd.notna(b):
                    # 増加なら +1、減少なら -1、変化なしなら 0
                    directions.append(np.sign(a - b))

            if not directions:
                return 0  # 比較できるペアがない場合は中立とする

            # 合計符号の sign → 全体傾向の方向
            return int(np.sign(sum(directions)))

        df_worker_lag['hours_trend_mode_6'] = df_worker_lag.apply(direction_mode, axis=1, args=(6,))


        return df_worker_lag

In [ ]:
# df_worker_lag = OvertimeWorkByMonthTimeseriesFeature().create_feature()

In [ ]:
df_worker_lag

In [ ]:
df_overtime = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_overtime_work_by_month.pkl"))

In [ ]:
df_overtime

In [ ]:
def make_worker_hours_lag_features(df_overtime, lag=35):
    """
    社員別の過去労働時間（lag特徴量）を作成し、最新月の1行にまとめる。

    Parameters:
        df_overtime: DataFrame
            '社員番号', 'date', 'hours' を含むDataFrame（dateは昇順が前提）
        lag: int
            生成する最大lag数（例：35であれば hours_1_age ～ hours_35_age）

    Returns:
        df_worker_lag: DataFrame
            社員番号ごとの最新行 + lag特徴量（hours_0_age ～ hours_{lag}_age）
    """
    df = df_overtime.copy()
    df = df.sort_values(['社員番号', 'date']).reset_index(drop=True)

    # lag特徴量を生成
    for i in range(1, lag + 1):
        df[f'hours_{i}_age'] = df.groupby('社員番号')['hours'].shift(i)

    # 最新行を抽出
    df_worker_lag = df.groupby('社員番号').tail(1).reset_index(drop=True)

    # カラム整形
    lag_cols = [f'hours_{i}_age' for i in range(1, lag + 1)]
    df_worker_lag = df_worker_lag[['社員番号', 'date', 'hours'] + lag_cols]
    df_worker_lag = df_worker_lag.rename(columns={'hours': 'hours_0_age'})

    return df_worker_lag
df_worker_lag = make_worker_hours_lag_features(df_overtime, lag=35)
df_worker_lag.drop('date', axis=1, inplace=True)
df_worker_lag.head()

In [ ]:
from sklearn.linear_model import LinearRegression

# 各種統計特徴量を生成するウィンドウサイズのリスト
windows = [2, 3, 4, 5, 6, 9, 12, 15, 18, 24, 36]
current_hours = df_worker_lag['hours_0_age']

for w in windows:
    print(w)

    # 直近 w ヶ月分の hours列（hours_0_age ～ hours_{w-1}_age）
    cols = [f'hours_{i}_age' for i in range(0, w)]

    # 移動統計量を計算
    df_worker_lag[f'hours_ma_{w}'] = df_worker_lag[cols].mean(axis=1)              # 平均
    df_worker_lag[f'hours_std_{w}'] = df_worker_lag[cols].std(axis=1)              # 標準偏差
    df_worker_lag[f'hours_max_{w}'] = df_worker_lag[cols].max(axis=1)              # 最大値
    df_worker_lag[f'hours_min_{w}'] = df_worker_lag[cols].min(axis=1)              # 最小値
    df_worker_lag[f'hours_diff_ma_{w}'] = current_hours - df_worker_lag[f'hours_ma_{w}']  # 今月と平均の差
    df_worker_lag[f'hours_range_{w}'] = df_worker_lag[f'hours_max_{w}'] - df_worker_lag[f'hours_min_{w}']  # 振れ幅
    df_worker_lag[f'hours_missing_count_{w}'] = df_worker_lag[cols].isna().sum(axis=1)  # 欠損数
    df_worker_lag[f'hours_zscore_{w}'] = (current_hours - df_worker_lag[f'hours_ma_{w}']) / (df_worker_lag[f'hours_std_{w}'] + 1e-6)  # z-score

    # 今月と wヶ月前との比較（差分）
    df_worker_lag[f'hours_diff_prev_{w}'] = current_hours - df_worker_lag[f'hours_{w-1}_age']

    # 線形トレンド（回帰直線の傾き）を算出
    trends = []
    for _, row in df_worker_lag[cols].iterrows():
        y = row.values
        x = np.arange(1, w + 1).reshape(-1, 1)
        if np.isnan(y).all():
            trends.append(np.nan)
        else:
            mask = ~np.isnan(y)
            reg = LinearRegression().fit(x[mask], y[mask])
            trends.append(reg.coef_[0])
    df_worker_lag[f'hours_trend_{w}'] = trends

    # 今月が過去平均より ±30% を超えているか
    ratio = current_hours / (df_worker_lag[f'hours_ma_{w}'] + 1e-6)
    df_worker_lag[f'hours_over_{w}_flag'] = (ratio > 1.3).astype(int)   # 今月が30%以上多い
    df_worker_lag[f'hours_under_{w}_flag'] = (ratio < 0.7).astype(int)  # 今月が30%以上少ない

    # 指数移動平均（最近の値をより重視した平均）
    df_worker_lag[f'hours_ewm_{w}'] = df_worker_lag[cols].T.ewm(span=3, axis=0).mean().iloc[-1]
    
df_worker_lag.head()

In [ ]:
df_worker_lag.head()

In [ ]:
# 前月との労働時間差が閾値を超えた回数（35ヶ月分で）
spike_threshold = 20
spike_count = []

for _, row in df_worker_lag.iterrows():
    diffs = []
    for i in range(1, 35):
        col_now = f'hours_{i}_age'
        col_next = f'hours_{i+1}_age'
        if pd.notna(row[col_now]) and pd.notna(row[col_next]):
            if abs(row[col_now] - row[col_next]) > spike_threshold:
                diffs.append(1)
    spike_count.append(sum(diffs))

df_worker_lag['hours_spike_count'] = spike_count
df_worker_lag.head()

In [ ]:
def direction_mode(row, window=6):
    """
    過去 window ヶ月分の労働時間（hours_{i}_age）を比較し、
    増減方向の多数決に基づいてトレンド方向を判定する。

    返す値:
        1  → 全体として上昇傾向
       -1  → 全体として下降傾向
        0  → 増減バラバラ or 相殺（中立）

    Parameters:
        row: pd.Series
        window: int (比較対象の月数)

    Returns:
        int: -1 / 0 / 1（下降 / 中立 / 上昇）
    """
    directions = []
    for i in range(0, window - 1):
        a, b = row.get(f'hours_{i}_age'), row.get(f'hours_{i+1}_age')
        if pd.notna(a) and pd.notna(b):
            # 増加なら +1、減少なら -1、変化なしなら 0
            directions.append(np.sign(a - b))

    if not directions:
        return 0  # 比較できるペアがない場合は中立とする

    # 合計符号の sign → 全体傾向の方向
    return int(np.sign(sum(directions)))

df_worker_lag['hours_trend_mode_6'] = df_worker_lag.apply(direction_mode, axis=1, args=(6,))

In [ ]:
df_worker_lag.head()

In [ ]:
df_worker_lag.head()

In [ ]:
df_worker_lag.shape

In [ ]:
class PositionHistoryFeature(FeatureBase):
    """
    PositionHistoryFeatureクラスは、役職履歴データを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        役職履歴データを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 役職履歴データを読み込む
        df_position_history = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_position_history.pkl"))

        df_position_history_feature = df_position_history.copy()[self.key_column].drop_duplicates()

        # # 基礎特徴量
        # df_position_history_base = df_position_history.groupby(self.key_column).agg(
        #     nunique_役職=('役職', 'nunique'),
        # ).reset_index()

        # # yearごとの役職
        # le = LabelEncoder()
        # df_position_history_ = df_position_history.copy()
        # df_position_history_['役職'] = le.fit_transform(df_position_history_['役職'])
        # for year in [22, 23, 24]:
        #     df_year_position = df_position_history_[df_position_history_['year'] == year][self.key_column + ['役職']]
        #     df_year_position.rename(columns={'役職': f'役職_{year}'}, inplace=True)
        #     df_position_history_base = df_position_history_base.merge(df_year_position, on=self.key_column, how='left')

        # 勤務区分ごとの年数を集計
        df_work_type_count = df_position_history.pivot_table(
            index='社員番号',
            columns="勤務区分",
            values="year",
            aggfunc="count",
            fill_value=None,
        ).reset_index()
        # カラム名の変更
        prefix = "ph_勤務区分_"
        df_work_type_count.columns = [col if col=='社員番号' else prefix + col for col in df_work_type_count.columns]

        # 役職ごとの年数を集計
        df_position_count = df_position_history.pivot_table(
            index='社員番号',
            columns="役職",
            values="year",
            aggfunc="count",
            fill_value=None,
        ).reset_index()
        # カラム名の変更
        prefix = "ph_役職_"
        df_position_count.columns = [col if col=='社員番号' else prefix + col for col in df_position_count.columns]

        # # 特徴量例: 各社員の役職変更回数
        # df_position_history_feature = df_position_history.groupby(self.key_column).agg(
        #     position_change_count=('役職', 'nunique'),
        #     first_position=('役職', 'first'),
        #     last_position=('役職', 'last'),
        # ).reset_index()

        # # one-hotエンコーディング(OneHotEncoder)
        # df_position_history_feature = one_hot_encode(df_position_history_feature, 'first_position')
        # df_position_history_feature = one_hot_encode(df_position_history_feature, 'last_position')

        # マージ
        # df_position_history_feature = df_position_history_feature.merge(df_position_history_base, on=self.key_column, how='left')
        df_position_history_feature = df_position_history_feature.merge(df_work_type_count, on=self.key_column, how='left')
        df_position_history_feature = df_position_history_feature.merge(df_position_count, on=self.key_column, how='left')

        return df_position_history_feature

# 処理実行

In [ ]:
def run_blocks(feature_blocks):
    print('start run blocks...')
    with Timer(prefix='run test'):
        for block in feature_blocks:
            with Timer(prefix='\t- {}'.format(str(block))):
                feature = block.create_feature()

In [ ]:
feature_blocks = [
    Key(use_cache=False, save_cache=True, logger=None),
	Target(use_cache=False, save_cache=True, logger=None),
    CategoryFeature(use_cache=False, save_cache=True, logger=None),
	CareerFeature(use_cache=False, save_cache=True, logger=None),
	UdemyActivityFeature(use_cache=False, save_cache=True, logger=None),
    UdemyEmbedding(use_cache=False, save_cache=True, logger=None),
	DxFeature(use_cache=False, save_cache=True, logger=None),
	HrFeature(use_cache=False, save_cache=True, logger=None),
	OvertimeWorkByMonthFeature(use_cache=False, save_cache=True, logger=None),
    OvertimeWorkByMonthTimeseriesFeature(use_cache=False, save_cache=True, logger=None),
	PositionHistoryFeature(use_cache=False, save_cache=True, logger=None),
]

In [ ]:
run_blocks(feature_blocks)

In [ ]:
print(Util.load_feature('CareerFeature').shape)
print(Util.load_feature('UdemyActivityFeature').shape)
print(Util.load_feature('UdemyEmbedding').shape)
print(Util.load_feature('DxFeature').shape) 
print(Util.load_feature('HrFeature').shape)
print(Util.load_feature('OvertimeWorkByMonthFeature').shape)
print(Util.load_feature('OvertimeWorkByMonthTimeseriesFeature').shape)
print(Util.load_feature('PositionHistoryFeature').shape)
print(Util.load_feature('Key').shape)
print(Util.load_feature('Target').shape)
print(Util.load_feature('CategoryFeature').shape)